In [ ]:
import pandas as pd
import pulp
import time
from pathlib import Path

In [ ]:
file_path = Path("../data/munchee_optimization_dataset.xlsx")

products_df = pd.read_excel(file_path, sheet_name="Products")
inventory_df = pd.read_excel(file_path, sheet_name="Inventory")
demand_df = pd.read_excel(file_path, sheet_name="Weekly_Demand")
shops_df = pd.read_excel(file_path, sheet_name="Shops")
vehicles_df = pd.read_excel(file_path, sheet_name="Vehicles")

for df in [products_df, inventory_df, demand_df, shops_df, vehicles_df]:
    df.columns = df.columns.str.strip().str.lower()

In [3]:
products = products_df["product_id"].tolist()
shops = shops_df["shop_id"].tolist()
vehicles = vehicles_df["vehicle_id"].tolist()

opening_stock = dict(zip(inventory_df["product_id"], inventory_df["opening_stock_cartons"]))
safety_stock = dict(zip(inventory_df["product_id"], inventory_df["safety_stock_cartons"]))

usable_stock = {
    p: max(opening_stock[p] - safety_stock[p], 0)
    for p in products
}

weights = dict(zip(products_df["product_id"], products_df["carton_weight_kg"]))
volumes = dict(zip(products_df["product_id"], products_df["carton_volume_m3"]))
margins = dict(zip(products_df["product_id"], products_df["gross_margin_lkr"]))

vehicle_carton_capacity = dict(zip(vehicles_df["vehicle_id"], vehicles_df["max_cartons"]))
vehicle_weight_capacity = dict(zip(vehicles_df["vehicle_id"], vehicles_df["max_payload_kg"]))
vehicle_volume_capacity = dict(zip(vehicles_df["vehicle_id"], vehicles_df["max_volume_m3"]))

total_carton_capacity = sum(vehicle_carton_capacity.values())
total_weight_capacity = sum(vehicle_weight_capacity.values())
total_volume_capacity = sum(vehicle_volume_capacity.values())

demand = {}
priority = {}

for _, row in demand_df.iterrows():
    demand[(row["shop_id"], row["product_id"])] = row["weekly_demand_cartons"]
    priority[(row["shop_id"], row["product_id"])] = row["priority_rank"]

In [14]:
model = pulp.LpProblem("Munchee_Allocation_ILP_Weighted", pulp.LpMaximize)

x = pulp.LpVariable.dicts(
    "alloc",
    [(s, p) for s in shops for p in products],
    lowBound=0,
    cat="Integer"
)

model += pulp.lpSum(
    ((6 - priority[(s, p)]) * 10 + margins[p]) * x[(s, p)]
    for s in shops for p in products
)

In [15]:
for s in shops:
    for p in products:
        model += x[(s, p)] <= demand.get((s, p), 0), f"demand_{s}_{p}"

In [16]:
for p in products:
    model += pulp.lpSum(x[(s, p)] for s in shops) <= usable_stock[p], f"stock_{p}"

In [17]:
model += pulp.lpSum(x[(s, p)] for s in shops for p in products) <= total_carton_capacity, "carton_capacity"

In [18]:
model += pulp.lpSum(weights[p] * x[(s, p)] for s in shops for p in products) <= total_weight_capacity, "weight_capacity"

In [19]:
model += pulp.lpSum(volumes[p] * x[(s, p)] for s in shops for p in products) <= total_volume_capacity, "volume_capacity"

In [20]:
start = time.time()
model.solve()
end = time.time()

runtime_ilp = end - start

print("Status:", pulp.LpStatus[model.status])
print("Runtime:", round(runtime_ilp, 4), "seconds")
print("Objective:", pulp.value(model.objective))

Status: Optimal
Runtime: 0.0738 seconds
Objective: 813080.0


In [21]:
allocation_rows = []

for s in shops:
    for p in products:
        qty = x[(s, p)].varValue
        if qty is not None and qty > 0:
            allocation_rows.append({
                "shop_id": s,
                "product_id": p,
                "allocated_cartons": int(qty)
            })

ilp_alloc_df = pd.DataFrame(allocation_rows)
ilp_alloc_df.head()

,shop_id,product_id,allocated_cartons
0,S001,P03,14
1,S001,P04,17
2,S001,P05,12
3,S001,P07,12
4,S001,P10,8


In [22]:
Path("../results/tables").mkdir(parents=True, exist_ok=True)
ilp_alloc_df.to_csv("../results/tables/ilp_allocation_results.csv", index=False)

In [25]:
total_demand = sum(demand.values())
fulfilled_demand = ilp_alloc_df["allocated_cartons"].sum()
unmet_demand = total_demand - fulfilled_demand
fill_rate = (fulfilled_demand / total_demand) * 100

used_stock_by_product = ilp_alloc_df.groupby("product_id")["allocated_cartons"].sum().to_dict()
used_stock_total = sum(used_stock_by_product.values())
total_usable_stock = sum(usable_stock.values())
stock_utilization = (used_stock_total / total_usable_stock) * 100 if total_usable_stock > 0 else 0

print("Total Demand:", total_demand)
print("Fulfilled Demand:", fulfilled_demand)
print("Unmet Demand:", unmet_demand)
print("Fill Rate %:", round(fill_rate, 2))
print("Stock Utilization %:", round(stock_utilization, 2))

Total Demand: 4082
Fulfilled Demand: 1440
Unmet Demand: 2642
Fill Rate %: 35.28
Stock Utilization %: 43.33
